In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

from agora.workflows import (
    build_persona_agent_configs,
    load_debate_construction,
    load_persona_catalog,
    load_question_catalog,
    load_prompt_catalog,
    run_debate_session,
)

# Persona debate configuration
Configure participant personas, question, and runtime controls.

In [ ]:
# Load the available configurations
prompt_path = Path('../data/prompts.json')
personas_path = Path('../data/personas.json')
questions_path = Path('../data/questions.json')
debate_construction_path = Path('../data/debate_construction.json')

personas = load_persona_catalog(personas_path)
questions = load_question_catalog(questions_path)
prompt_catalog = load_prompt_catalog(prompt_path)
debate_construction = load_debate_construction(debate_construction_path)

In [ ]:
import json
import shutil
from agora.debate_analyzer import DebateAnalyzer

def run_experiment(
    scenario_id: str,
    prompt_set: str = "default",
    question_variant: str = "controversial",
    side_order: str = "12",
    debate_arena: str = "neutral",
    alpha_model: str = "anthropic/claude-sonnet-4.5", # "openai/gpt-4o-mini",
    beta_model: str = "anthropic/claude-sonnet-4.5",
    turns_per_agent: int = 10,
    verbose: bool = True,
    show_plots: bool = True,
):
    """
    Run a complete debate experiment with evaluation.
    
    Args:
        scenario_id: ID from debate_construction["pair_scenarios"]
        question_index: Which question within the scenario (0-based)
        question_variant: "agreeable" or "controversial"
        side_order: "12" (side_1=alpha) or "21" (side_2=alpha)
        debate_arena: "neutral" or "bias" (uses alpha's arena)
        alpha_model: Model for alpha agent
        beta_model: Model for beta agent
        turns_per_agent: Number of debate turns per agent
        verbose: Print debate output
        show_plots: If True, display plots in notebook; if False, just save to file
    """
    # === Parse scenario ===
    scenarios = debate_construction.get("pair_scenarios", [])
    scenario = next((s for s in scenarios if s.get("id") == scenario_id), None)
    if scenario is None:
        raise ValueError(f"Scenario '{scenario_id}' not found")
    
    # Scenario has single question_id (not question_ids list)
    question_id = scenario.get("question_id", "")
    
    # Map sides
    s1, s2 = scenario["side_1_persona_id"], scenario["side_2_persona_id"]
    alpha_persona_id, beta_persona_id = (s1, s2) if side_order == "12" else (s2, s1)
    
    # Arena override
    debate_arena_override = prompt_catalog[prompt_set]["neutral_arena_prompt"] if debate_arena == "neutral" else None
    arena_display = "neutral" if debate_arena == "neutral" else f"bias ({alpha_persona_id})"
    
    print(f"[run] scenario={scenario_id}  q={question_id}/{question_variant}  alpha={alpha_persona_id}  beta={beta_persona_id}  arena={arena_display}")
    
    # TODO no hardcoded paths are allowed (snapshots, evaluation results). These should be set outside the function.
    # === Paths ===
    run_name = f'{scenario_id}_{question_id}_{question_variant}_{side_order}_{debate_arena}'
    snapshot_dir = Path('../outputs/snapshots')
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    snapshot_path = snapshot_dir / f'{run_name}.json'
    
    eval_output_dir = Path('../outputs/evaluation_results') / run_name
    eval_output_dir.mkdir(parents=True, exist_ok=True)
    
    # === Run debate ===
    agent_configs = build_persona_agent_configs(
        alpha_persona_id=alpha_persona_id,
        beta_persona_id=beta_persona_id,
        question_id=question_id,
        question_variant=question_variant,
        debate_arena_override=debate_arena_override,
        personas=personas,
        questions=questions,
        alpha_model=alpha_model,
        beta_model=beta_model,
        prompt_set=prompt_set,
        prompt_catalog=prompt_catalog,
        private_response_keep=False,
        pre_interview_keep=False,
        post_interview_keep=False,
    )
    
    agora, agents = run_debate_session(
        agent_configs,
        turns_per_agent=turns_per_agent,
        verbose=verbose,
        snapshot_path=snapshot_path,
        load_snapshot_flag=False,
        save_snapshot_flag=True,
        skip_first_agent_first_reflection=True,
    )
    
    # === Evaluation ===
    analyzer = DebateAnalyzer(agora.history())
    intra_scores = analyzer.compute_intra_agent_honesty()
    inter_external = analyzer.compute_inter_agent_alignment("public_speech", "public_speech")
    inter_internal = analyzer.compute_inter_agent_alignment("private_reflection", "private_reflection")
    
    # === Save plots ===
    alpha_name = personas['personas'][alpha_persona_id]['name']
    beta_name = personas['personas'][beta_persona_id]['name']
    label_map = {'Alpha': f'Alpha: {alpha_name}', 'Beta': f'Beta: {beta_name}'}
    
    # Intra-agent plot
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    for speaker, data in intra_scores.items():
        ax1.plot(data["turns"], data["scores"], marker='o', label=label_map.get(speaker, speaker))
    ax1.set_title(f'Intra-Agent Honesty - {question_id} - {question_variant}\nSide: {side_order} | Arena: {debate_arena}')
    ax1.set_xlabel('Debate Turn'); ax1.set_ylabel('Cosine Similarity'); ax1.set_ylim(0, 1)
    ax1.legend(); ax1.grid(); plt.tight_layout()
    fig1.savefig(eval_output_dir / 'intra_agent.png', dpi=150, bbox_inches='tight')
    if show_plots:
        plt.show()
    plt.close(fig1)
    
    # Inter-agent plot
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(inter_external["turns"], inter_external["scores"], marker='o', label='External (Public)')
    ax2.plot(inter_internal["turns"], inter_internal["scores"], marker='o', label='Internal (Private)')
    ax2.set_title(f'Inter-Agent Alignment - {question_id} - {question_variant}\nAlpha: {alpha_name} | Beta: {beta_name} | Side: {side_order} | Arena: {debate_arena}')
    ax2.set_xlabel('Debate Turn'); ax2.set_ylabel('Cosine Similarity'); ax2.set_ylim(0, 1)
    ax2.legend(); ax2.grid(); plt.tight_layout()
    fig2.savefig(eval_output_dir / 'inter_agent.png', dpi=150, bbox_inches='tight')
    if show_plots:
        plt.show()
    plt.close(fig2)
    
    # === Save data ===
    eval_data = {
        'metadata': {
            'scenario_id': scenario_id, 'question_id': question_id, 'question_variant': question_variant,
            'side_order': side_order, 'debate_arena': debate_arena,
            'alpha_persona_id': alpha_persona_id, 'alpha_persona_name': alpha_name,
            'beta_persona_id': beta_persona_id, 'beta_persona_name': beta_name,
        },
        'intra_agent_honesty': intra_scores,
        'inter_agent_alignment': {'external': inter_external, 'internal': inter_internal},
    }
    with open(eval_output_dir / 'eval_data.json', 'w') as f:
        json.dump(eval_data, f, indent=2)
    
    if snapshot_path.exists():
        shutil.copy(snapshot_path, eval_output_dir / 'debate_snapshot.json')
    
    print(f"\n✓ Saved to: {eval_output_dir}/")
    return agora, analyzer


In [ ]:
# ===== RUN ALL EXPERIMENTS =====
import itertools
from datetime import datetime

# Get all scenarios and their questions
all_scenarios = debate_construction.get("pair_scenarios", [])
variants = ["agreeable", "controversial"]
side_orders = ["12", "21"]
arenas = ["neutral", "bias"]

# Settings
TURNS_PER_AGENT = 2
VERBOSE = False  # Set True to see full debate output

# Build all combinations
# Filter to specific scenarios (or set to None to run all)
SELECTED_SCENARIOS = ["cold_war_deterrence"]

combinations = []
for scenario in all_scenarios:
    scenario_id = scenario["id"]
    if SELECTED_SCENARIOS is None or scenario_id in SELECTED_SCENARIOS:
        # Each scenario has one question_id (no need to loop over question_ids)
        for variant, side, arena in itertools.product(variants, side_orders, arenas):
            combinations.append((scenario_id, 0, variant, side, arena))

print(f"Total experiments: {len(combinations)}")
print(f"Scenarios: {[s['id'] for s in all_scenarios if (s['id'] in SELECTED_SCENARIOS) or (SELECTED_SCENARIOS is None)]}")
print(f"Variants: {variants}, Side orders: {side_orders}, Arenas: {arenas}")
print(f"\n{'='*60}")

# Run all experiments
results = []
start_time = datetime.now()

for i, (scenario_id, q_idx, variant, side, arena) in enumerate(combinations, 1):
    print(f"\n[{i}/{len(combinations)}] {scenario_id} q{q_idx} {variant} {side} {arena}")
    try:
        agora, analyzer = run_experiment(
            scenario_id=scenario_id,
            question_variant=variant,
            side_order=side,
            debate_arena=arena,
            turns_per_agent=TURNS_PER_AGENT,
            verbose=VERBOSE,
            show_plots=False,  # Don't display plots in batch mode
        )
        results.append({"combo": (scenario_id, q_idx, variant, side, arena), "success": True})
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        results.append({"combo": (scenario_id, q_idx, variant, side, arena), "success": False, "error": str(e)})

# Summary
duration = datetime.now() - start_time
print(f"\n{'='*60}")
print(f"COMPLETE! {sum(r['success'] for r in results)}/{len(results)} succeeded")
print(f"Total time: {duration}")
